# NB_01 — Absorber Manufacturing Synthesis v3

This notebook is now an orchestration layer.

Reusable synthesis logic lives in:

```text
tools/synthesis/
    concept_matcher.py
    specification_generator.py
    notebook_selector.py
```

Engineering vocabulary and rules remain in:

```text
engineering_navigator/synthesis/
    engineering_concepts.yaml
    synthesis_rules.yaml
```

The notebook loads source records, runs the reusable synthesis engine, writes outputs, and downloads the export ZIP.


## 1. Configuration and repository paths

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

import pandas as pd
import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE: str | Path | None = None

SOURCE_FILES = [
    "SOURCE_00_becker_transition_models.yaml",
    "SOURCE_01_bismuth_microstructure.yaml",
    "SOURCE_02_eliminating_nongaussian_spectral_response.yaml",
]
SYNTHESIS_ID = "SYNTHESIS_01"


def find_repo_root() -> Path:
    candidates = []

    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])
    candidates.extend(
        [
            Path("/content/sensors-becker"),
            Path("/home/dan/sensors-becker"),
            Path.home() / "sensors-becker",
        ]
    )

    for candidate in candidates:
        if candidate.is_dir() and (candidate / "engineering_navigator").is_dir():
            return candidate

    if Path("/content").exists():
        target = Path("/content/sensors-becker")
        if not target.exists():
            subprocess.run(
                ["git", "clone", REPOSITORY_URL, str(target)],
                check=True,
            )
        return target

    raise FileNotFoundError(
        "Could not locate sensors-becker. Set REPO_ROOT_OVERRIDE explicitly."
    )


REPO_ROOT = find_repo_root()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SOURCE_DIR = (
    REPO_ROOT
    / "engineering_navigator"
    / "absorber_manufacturing"
    / "source_records"
)
SYNTHESIS_DIR = REPO_ROOT / "engineering_navigator" / "synthesis"
CONCEPTS_FILE = SYNTHESIS_DIR / "engineering_concepts.yaml"
RULES_FILE = SYNTHESIS_DIR / "synthesis_rules.yaml"

OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
    / "engineering_questions"
    / "absorber_manufacturing"
    / SYNTHESIS_ID
)
EXPORT_DIR = REPO_ROOT / "exports" / SYNTHESIS_ID
EXPORT_ZIP = REPO_ROOT / "exports" / f"{SYNTHESIS_ID}_export.zip"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)

from tools.synthesis import (
    build_engineering_axis_matrix,
    collect_source_relationships,
    match_relationship_concepts,
    collect_unreported_gaps,
    generate_candidate_specifications,
    generate_open_specifications,
    select_next_notebook,
)

print(f"Repository : {REPO_ROOT}")
print(f"Sources    : {SOURCE_DIR.relative_to(REPO_ROOT)}")
print(f"Concepts   : {CONCEPTS_FILE.relative_to(REPO_ROOT)}")
print(f"Rules      : {RULES_FILE.relative_to(REPO_ROOT)}")


## 2. Load YAML inputs and source records

In [ ]:
def load_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing YAML: {path}")

    data = yaml.safe_load(path.read_text(encoding="utf-8"))

    if not isinstance(data, dict):
        raise TypeError(f"{path.name}: expected one top-level mapping")

    return data


concept_config = load_yaml(CONCEPTS_FILE)
rule_config = load_yaml(RULES_FILE)

records = {}
source_slots = {}

for index, filename in enumerate(SOURCE_FILES):
    record = load_yaml(SOURCE_DIR / filename)
    source_id = record.get("source_id")

    if not source_id:
        raise KeyError(f"{filename}: missing source_id")

    if source_id in records:
        raise ValueError(f"Duplicate source_id: {source_id}")

    records[source_id] = record
    source_slots[f"SOURCE_{index:02d}"] = source_id

print(f"Loaded {len(records)} source records.")
print(f"Loaded {len(concept_config.get('concepts', []))} relationship concepts.")
print(f"Loaded {len(rule_config.get('candidate_specifications', []))} candidate-spec rules.")


## 3. Validate source-record status

In [ ]:
status_rows = []

for source_id, record in records.items():
    status_rows.append(
        {
            "source_id": source_id,
            "title": record.get("title", ""),
            "record_status": record.get("record_status", ""),
            "extraction_status": record.get("extraction_status", ""),
            "reported_values": len(record.get("reported_values", [])),
            "relationships": len(record.get("engineering_relationships", [])),
        }
    )

status_df = (
    pd.DataFrame(status_rows)
    .sort_values("source_id")
    .reset_index(drop=True)
)

incomplete = status_df[
    ~status_df["extraction_status"].astype(str).str.startswith("complete")
]

if not incomplete.empty:
    raise ValueError(
        "All source records must be complete before synthesis:\n"
        + incomplete[["source_id", "extraction_status"]].to_string(index=False)
    )

print("Source-record validation: PASS")
status_df


## 4. Run reusable concept-matching engine

In [ ]:
variable_matrix = build_engineering_axis_matrix(
    records,
    concept_config.get("engineering_axes", []),
)

source_relationships_df = collect_source_relationships(
    records,
    concept_config.get("concepts", []),
)

relationships_df = match_relationship_concepts(
    source_relationships_df,
    concept_config.get("concepts", []),
)

relationships_df


## 5. Collect quantitative evidence

In [ ]:
value_rows = []

for source_id, record in records.items():
    for item in record.get("reported_values", []):
        if not isinstance(item, dict):
            continue

        value_rows.append(
            {
                "source_id": source_id,
                "object": item.get("object"),
                "variable": item.get("variable"),
                "value": item.get("value"),
                "unit": item.get("unit"),
                "condition": item.get("condition"),
                "source_page": item.get("source_page"),
            }
        )

values_df = pd.DataFrame(value_rows)

FOCUS_VARIABLES = {
    "Tc",
    "Bi_thickness",
    "C",
    "G",
    "SEM_grain_size",
    "diffraction_grain_size",
    "average_grain_size",
    "average_grain_radius",
    "quantum_efficiency",
    "residual_resistance_ratio",
    "cloud_size",
    "delta_E",
    "predicted_delta_E",
}

focus_values = (
    values_df[values_df["variable"].isin(FOCUS_VARIABLES)]
    .sort_values(["variable", "source_id", "object"])
    .reset_index(drop=True)
)

focus_values


## 6. Generate candidate and open specifications

In [ ]:
candidate_specifications, specifications_df = generate_candidate_specifications(
    relationships_df,
    rule_config.get("candidate_specifications", []),
    spec_prefix="SPEC_AM",
)

gaps_df = collect_unreported_gaps(records)

open_items, open_specs_df = generate_open_specifications(
    gaps_df,
    rule_config.get("open_specifications", []),
)

print(f"Candidate specifications: {len(candidate_specifications)}")
print(f"Open specifications     : {len(open_items)}")

specifications_df


## 7. Select next engineering notebook

In [ ]:
next_notebook = select_next_notebook(
    open_specs_df,
    rule_config.get("next_notebooks", []),
    SOURCE_FILES,
)

next_notebook


## 8. Write synthesis outputs

In [ ]:
status_csv = OUTPUT_DIR / "source_status.csv"
variable_matrix_csv = OUTPUT_DIR / "variable_matrix.csv"
focus_values_csv = OUTPUT_DIR / "quantitative_evidence.csv"
source_relationships_csv = OUTPUT_DIR / "source_relationships.csv"
relationships_csv = OUTPUT_DIR / "synthesis_relationships.csv"
specifications_csv = OUTPUT_DIR / "candidate_specifications.csv"
open_specs_csv = OUTPUT_DIR / "open_specifications.csv"
synthesis_json = OUTPUT_DIR / "synthesis_summary.json"

status_df.to_csv(status_csv, index=False)
variable_matrix.to_csv(variable_matrix_csv, index=False)
focus_values.to_csv(focus_values_csv, index=False)
source_relationships_df.to_csv(source_relationships_csv, index=False)
relationships_df.to_csv(relationships_csv, index=False)
specifications_df.to_csv(specifications_csv, index=False)
open_specs_df.to_csv(open_specs_csv, index=False)

synthesis_summary = {
    "synthesis_id": SYNTHESIS_ID,
    "sources": sorted(records),
    "source_slots": source_slots,
    "relationship_concepts": relationships_df.to_dict("records"),
    "candidate_specifications": candidate_specifications,
    "open_specifications": open_items,
    "next_notebook": next_notebook,
}

synthesis_json.write_text(
    json.dumps(synthesis_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

written_files = {
    "source_status": status_csv,
    "variable_matrix": variable_matrix_csv,
    "quantitative_evidence": focus_values_csv,
    "source_relationships": source_relationships_csv,
    "synthesis_relationships": relationships_csv,
    "candidate_specifications": specifications_csv,
    "open_specifications": open_specs_csv,
    "synthesis_summary": synthesis_json,
}

for name, path in written_files.items():
    print(f"{name:26} {path.relative_to(REPO_ROOT)}")


## 9. Build and download export ZIP

In [ ]:
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for path in written_files.values():
    shutil.copy2(path, EXPORT_DIR / path.name)

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

with zipfile.ZipFile(
    EXPORT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(EXPORT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print(f"Export package: {EXPORT_ZIP}")
print(f"Size: {EXPORT_ZIP.stat().st_size:,} bytes")

try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Google Colab.")


## 10. Handoff

The reusable synthesis engine is now available to other engineering drivers.

To reuse it elsewhere:

1. supply completed source records;
2. point the notebook at an engineering-concepts YAML and synthesis-rules YAML;
3. call the reusable modules;
4. write driver-specific outputs.

*Admissible generalizations trail leading specifications.*
